# Vibronic coupling in tetracene dimers
## General post-processing / prototyping notebook
### Can be done locally:


1. Boys localisation and partial diagonalisation (requires calc.chk)
    - reconstruct Fock matrix (takes some time)
    - localize frontier orbitals
    - visualize frontier orbitals (careful with notebook size)


2. Electron-phonon coupling analysis (requires out.h5)
    - transform e-ph coupling to frontier orbital space
    - visualize molecular distortions
    - calculate effective exchange vibronic coupling in the TT subspace from two-electron integrals and one-electron LVC [Kollmar, J. Chem. Phys. 98 (1993)]
    - reduce dimensionality of the e-ph coupling (clustering, SVD)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
import py3Dmol
from pyscf import gto, dft
from dftutils import mf_from_chk, Boys_frontier_orbitals, \
                     pair_neighboring_orbitals, block_frontier_orbitals, \
                     fock_matrix, two_electron_integrals, save_orbital
from visutils import view_orbital

HARTREE2EV = 27.2114079527
HARTREE2ICM = 219474.63

# Directories for source data and results
DATADIR = os.path.realpath("../runs/10264877/")
SAVEDIR = os.path.realpath("../results/")

## 1. Import and post-process DFT calculation

Import ```mf``` and ```mol``` object from checkfile.

In [ ]:
mf = mf_from_chk(DATADIR + "/" + "calc.chk")
mol = mf.mol

Calculate Fock matrix in the atomic orbital basis (time consuming, takes about 2 minutes for a tetracene dimer with cc-pvdz).

In [ ]:
fock_ao = fock_matrix(mf)

Localize the 4 frontier molecular orbitals using Boys localisation. We expect two Boys orbitals on each tetracene fragment. Reorder the molecular orbitals so that they appear one pair after another. After that, block-diagonalize the Fock matrix to obtain localized orthogonal orbitals corresponding to HOMO and LUMO of each tetracene unit.

In [ ]:
mo_coeff_boys, idx = Boys_frontier_orbitals(mf, 4)
mo_coeff_boys, idx = pair_neighboring_orbitals(mf, mo_coeff_boys)
mo_coeff_block = block_frontier_orbitals(mf, mo_coeff_boys, fock=fock_ao)

Fock matrix in the frontier subspace in different bases: MO, Boys, block-diagonal

In [ ]:
fock_mo = mf.mo_coeff[:,idx].T @ fock_ao @ mf.mo_coeff[:,idx]
fock_boys = mo_coeff_boys.T @ fock_ao @ mo_coeff_boys
fock_block = mo_coeff_block.T @ fock_ao @ mo_coeff_block
print(fock_mo * HARTREE2EV * 1000, '\n')
print(fock_boys * HARTREE2EV * 1000, '\n')
print(fock_block * HARTREE2EV * 1000)

### Save and visualize molecular orbitals
Save orbitals as ```.cube``` and ```.html``` files. These can be rendered by py3Dmol inside a Jupyter notebook, or opened in an external browser. Visualization in an external browser helps keep the notebook nice and lightweight (strongly recommended if tracked by git).

In [ ]:
# Grid for cube file: low-res (100,50,50), hi-res (400,100,100) 
grid = (100, 50, 50)

# Boys orbitals
for i, mo in enumerate(mo_coeff_boys.T):
    orbname = SAVEDIR + "/" + "orbBoys" + str(i+1) + ".cube"
    save_orbital(mol, orbname, mo, grid=grid)
    v = view_orbital(mol, orbname, iso=0.02, alpha=0.9, html=True)

# Block-diagonalized orbitals
for i, mo in enumerate(mo_coeff_block.T):
    orbname = SAVEDIR + "/" + "orbBlock" + str(i+1) + ".cube"
    save_orbital(mol, orbname, mo, grid=grid)
    v = view_orbital(mol, orbname, iso=0.02, alpha=0.9, html=True)

### Estimate TT exchange coupling
Follow Kollmar 1993 and estimate TT exchange as a energy shift from CI between TT and CT states as
$$ J = \frac{t^2}{U+K}. $$

Calculate two-electron integrals in the frontier subspace ```integral2e(i,j,k,l)```, corresponding to
$( i j | k l ) = \langle i k | j l \rangle $.
This is quite time consuming (takes approximately 5 minutes for a tetracene dimer with cc-pvdz).

In [ ]:
integral2e = two_electron_integrals(mol, mo_coeff_block)

In [ ]:
J_hAlA = integral2e[0,0,1,1] * HARTREE2EV * 1000
J_hBlB = integral2e[2,2,3,3] * HARTREE2EV * 1000
K_hAlA = integral2e[0,1,1,0] * HARTREE2EV * 1000
K_hBlB = integral2e[2,3,3,2] * HARTREE2EV * 1000
print(J_hAlA, J_hBlB)
print(K_hAlA, K_hBlB)

Save DFT post-processing results

In [ ]:
hdf5_file_path = DATADIR + "/" + "loc.h5"
with h5py.File(hdf5_file_path, 'w') as hdf5_file:
    hdf5_file.create_dataset("fock_ao", data=fock_ao)
    hdf5_file.create_dataset("mo_coeff_block", data=mo_coeff_block)
    hdf5_file.create_dataset("integral2e", data=integral2e)


According to Smith & Michl, Chem. Rev. 110, 6891-6936 (2010), the CI Hamiltonian matrix elements within the subspace spanned by singlet excitons (SE), charge transfer states (CT), and singlet triplet-triplet (TT) are
$$
\begin{align}
\langle \mathrm{CT}_A|H|\mathrm{CT}_B\rangle &= 2(h_Al_B|l_Ah_B)-(h_Ah_B|l_Al_B) \\
\langle \mathrm{SE}_A|H|\mathrm{SE}_B\rangle &= 2(h_Al_A|l_Bh_B)-(h_Ah_B|l_Bl_A) \\
\langle \mathrm{CT}_A|H|\mathrm{SE}_A\rangle &= \langle l_A|F|l_B \rangle+2(h_Al_B|l_Ah_A)-(h_Ah_A|l_Al_B) \\
\langle \mathrm{CT}_A|H|\mathrm{SE}_B\rangle &= \langle h_A|F|h_B \rangle+2(h_Bl_B|l_Bh_A)-(h_Bh_A|l_Bl_B) \\
\langle \mathrm{SE}_A|H|\mathrm{TT}\rangle &= \sqrt{\frac{3}{2}} \left[(l_Ah_B|l_Bl_A)-(h_Al_B|h_Bh_A)\right]\\
\langle \mathrm{CT}_A|H|\mathrm{TT}\rangle &= \sqrt{\frac{3}{2}} \left[ \langle l_A|F|h_B\rangle + (l_Ah_B|l_Bl_B)-(l_Ah_B|h_Ah_A) \right]
\end{align}
$$
(all other matrix elements can be obtained by switching $A\leftrightarrow B$) and using hermiticity)

In [2]:
# Import DFT data
mf = mf_from_chk(DATADIR + "/" + "calc.chk")
mol = mf.mol
h5file = DATADIR + "/" + "loc.h5"
with h5py.File(h5file, "r") as f:
    fock_ao = np.array(list(f["fock_ao"])) # hartree
    mo_coeff_block = np.array(list(f["mo_coeff_block"]))
    integral2e = np.array(list(f["integral2e"]))

In [27]:
grid = (200, 100, 100)
mo_coeff_boys, idx = Boys_frontier_orbitals(mf, norb=4)
mo_coeff_front = mf.mo_coeff[:, idx]

# frontier orbitals
for i in range(8):
    mo = mo_coeff_front[:,i]
    orbname = SAVEDIR + "/" + "orbFront" + str(i+1) + ".cube"
    save_orbital(mol, orbname, mo, grid=grid)
    view_orbital(mol, orbname, iso=0.02, alpha=0.9, html=True)

IndexError: index 4 is out of bounds for axis 1 with size 4

In [24]:
# boys orbitals
for i in range(4):
    mo = mo_coeff_boys[:,i]
    orbname = SAVEDIR + "/" + "orbBoys" + str(i+1) + ".cube"
    #save_orbital(mol, orbname, mo, grid=grid)
    view_orbital(mol, orbname, iso=0.02, alpha=0.9, html=True)

In [25]:
# block-diagonalised orbitals
for i in range(4):
    mo = mo_coeff_boys[:,i]
    orbname = SAVEDIR + "/" + "orbBlock" + str(i+1) + ".cube"
    #save_orbital(mol, orbname, mo, grid=grid)
    view_orbital(mol, orbname, iso=0.02, alpha=0.9, html=True)